# PerturbFlow walkthrough: synthetic fixture (Replogle 2022-style screen)

This notebook walks through the PerturbFlow pipeline end-to-end on the
package's built-in synthetic fixture — 2000 cells, 10 guides, 5
perturbations — designed to mimic a Replogle 2022 essential-gene
CRISPRi screen at small scale.

**To swap in real Replogle 2022 data:** the public release is at
[GEO GSE177150](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE177150)
and the processed gene-program AnnDatas are on
[figshare](https://plus.figshare.com/articles/dataset/Genome-scale_Perturb-seq/20029387).
Replace the `build_synthetic()` call below with `pf.read_10x_h5(...)`
and `pf.read_guide_calls(...)` pointing at the downloaded files.

## 1. Set up

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

import perturbflow as pf
from perturbflow.config import (
    DEConfig,
    DownstreamConfig,
    GuideAssignmentConfig,
    InputConfig,
    PerturbationAnalysisConfig,
    QCConfig,
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

sc.settings.verbosity = 1
sns.set_theme(style='whitegrid', context='notebook')

print(f'PerturbFlow version: {pf.__version__}')

## 2. Load data

For Replogle 2022, this is:
```python
adata = pf.read_10x_h5('replogle_2022/filtered_feature_bc_matrix.h5')
guide_calls = pf.read_guide_calls('replogle_2022/guide_calls.csv')
guide_metadata = pf.read_guide_metadata('replogle_2022/guide_library.csv')
```

In [ ]:
from tests.fixtures.make_synthetic import build_synthetic

bundle = build_synthetic(seed=0)
adata = bundle['adata']
guide_calls = bundle['guide_calls']
guide_metadata = bundle['guide_metadata']

print(adata)
print(f'Guide calls: {guide_calls.shape}')
print(f'Guides in library: {len(guide_metadata)}')

## 3. Guide assignment

Three-rule procedure: UMI floor, dominance ratio, multi-guide cap. See
[docs/methodology.md](../docs/methodology.md#2-guide-assignment) for
the rationale.

In [ ]:
adata = pf.assign_guides(
    adata,
    guide_calls,
    guide_metadata,
    config=GuideAssignmentConfig(
        min_guide_umi=5,
        dominance_ratio=2.0,
        max_guides=1,
        drop_unassigned=True,
    ),
)

summary = adata.obs['assignment_status'].astype(str).value_counts()
print(summary)
print(f'\nPerturbations: {sorted(adata.obs["perturbation"].astype(str).unique())}')

## 4. QC tables

PerturbFlow produces three QC tables. We'll look at the per-guide and
per-cell tables here; the per-perturbation table is more informative
after Mixscape (see step 6).

In [ ]:
per_cell = pf.per_cell_qc(adata)
per_guide = pf.per_guide_qc(adata, guide_metadata)

print('Per-cell QC summary:')
print(per_cell[['total_counts', 'n_genes']].describe())
print('\nPer-guide coverage:')
print(per_guide[['guide_id', 'target_gene', 'is_control', 'n_cells', 'mean_guide_umi']])

## 5. Cell-state landscape

Standard Scanpy UMAP, colored by perturbation. This is the figure-1
plot.

In [ ]:
tmp = adata.copy()
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)
sc.pp.scale(tmp, max_value=10)
sc.tl.pca(tmp, n_comps=30, random_state=0)
sc.pp.neighbors(tmp, random_state=0)
sc.tl.umap(tmp, random_state=0)
adata.obsm['X_pca'] = tmp.obsm['X_pca']
adata.obsm['X_umap'] = tmp.obsm['X_umap']
del tmp

sc.pl.umap(adata, color='perturbation', frameon=False, size=20, title='Cells coloured by perturbation')

## 6. Mixscape: separating KO from escaped cells

Even in well-validated screens, 10–40% of cells that received a guide
fail to perturb. Mixscape flags them so we can drop them from DE.

In [ ]:
adata = pf.run_mixscape(
    adata,
    config=PerturbationAnalysisConfig(
        control_label='NT',
        n_neighbors=20,
        mixscape_pval_cutoff=0.05,
    ),
)

if 'mixscape_class_global' in adata.obs:
    print(adata.obs['mixscape_class_global'].value_counts())
print(f'\nKO cells: {int(adata.obs["mixscape_perturbed"].sum())}')
print(f'NP cells: {int((~adata.obs["mixscape_perturbed"]).sum() - (adata.obs["perturbation"].astype(str) == "NT").sum())}')

In [ ]:
per_pert = pf.per_perturbation_qc(adata, control_label='NT')
per_pert

**Reading the per-perturbation table:**
- `escape_fraction` — what fraction of carrying cells Mixscape called NP
  (didn't perturb).
- `on_target_log2fc` — log2 fold-change of the target gene in cells
  carrying the guide vs control cells. Negative = effective knockdown.
  This is your single-number sanity check that the screen worked.

## 7. Pseudobulk DE

DESeq2 on pseudobulk counts, filtered to KO cells only (Mixscape's NP
cells are dropped from the treatment arm).

In [ ]:
de_results = pf.run_pseudobulk_de(
    adata,
    config=DEConfig(
        enable=True,
        min_replicates_per_group=2,
        min_cells_per_replicate=5,
        lfc_threshold=0.5,
        padj_threshold=0.1,
        use_mixscape_filter=True,
    ),
    input_config=InputConfig(n_pseudo_replicates=3),
    control_label='NT',
)

print(f'DE tested for {len(de_results)} perturbations:')
for pert, df in de_results.items():
    n_sig = int(df['significant'].sum()) if 'significant' in df.columns else 0
    on_target = df[df['gene'] == pert]
    if not on_target.empty:
        lfc = float(on_target['log2FoldChange'].iloc[0])
        padj = float(on_target['padj'].iloc[0])
        print(f'  {pert}: {n_sig} significant genes, on-target log2FC={lfc:+.2f} (padj={padj:.2e})')

In [ ]:
# Plot per-perturbation volcanos.
import numpy as np
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (pert, df) in zip(axes.flat, list(de_results.items())[:4]):
    nlogp = -np.log10(df['padj'].clip(1e-300))
    sig = df.get('significant', pd.Series(False, index=df.index)).astype(bool)
    ax.scatter(df['log2FoldChange'], nlogp, s=6, alpha=0.4, color='#9e9e9e', label='ns')
    ax.scatter(df.loc[sig, 'log2FoldChange'], nlogp[sig], s=10, alpha=0.85, color='#c62828', label='sig')
    if pert in df['gene'].values:
        row = df[df['gene'] == pert].iloc[0]
        ax.scatter(row['log2FoldChange'], -np.log10(max(row['padj'], 1e-300)), s=60, color='#1565c0', marker='*', label='on-target')
    ax.axhline(-np.log10(0.05), color='#90a4ae', linestyle='--', linewidth=0.7)
    ax.set_title(pert)
    ax.set_xlabel('log2 fold-change')
    ax.set_ylabel('-log10(padj)')
    ax.legend(fontsize=7)
fig.suptitle('Per-perturbation volcanos', y=1.02)
plt.tight_layout()
plt.show()

## 8. Cell-state effect map

Quantify how much each perturbation moves cells in UMAP space relative
to controls.

In [ ]:
cell_state = pf.compute_cell_state_effects(adata, control_label='NT')
cell_state

## 9. Pathway scoring (optional)

decoupler-py against the chosen network. Requires `pip install
perturbflow[pathways]` and an internet connection on first run (to fetch
the MSigDB collection).

In [ ]:
# Pathway scoring requires real gene symbols; the synthetic fixture's
# generic names won't return useful results. The call still runs and
# returns an empty frame if no genes match.
pathway_scores = pf.score_pathways(de_results, config=DownstreamConfig(
    enable_pathway_scoring=True,
    pathway_net='hallmarks',
    pathway_method='ulm',
))
print(f'Pathway score rows: {len(pathway_scores)}')
if not pathway_scores.empty:
    print(pathway_scores.head(10))

## 10. Render the report

Produces a self-contained HTML file you can share.

In [ ]:
from pathlib import Path
from perturbflow.provenance import collect as collect_prov
from perturbflow.config import PerturbFlowConfig

out = Path('notebook_results')
out.mkdir(exist_ok=True)

provenance = collect_prov(PerturbFlowConfig(), config_path=None)

report_path = pf.write_html_report(
    out_path=out / 'report.html',
    run_name='replogle_walkthrough',
    seed=0,
    version=pf.__version__,
    git_rev=str(provenance['git'].get('revision', 'unknown'))[:12],
    per_cell=per_cell,
    per_perturbation=per_pert,
    de_results=de_results,
    pathway_scores=pathway_scores,
    umap_coords=adata.obsm['X_umap'],
    umap_labels=adata.obs['perturbation'].astype(str),
    provenance=provenance,
)
print(f'Report: {report_path}')

## What's next

- The [methodology document](../docs/methodology.md) explains why each
  step does what it does.
- The [cookbook](../docs/cookbook.md) has 10 recipes for common
  real-world questions (multi-donor screens, time series, CRISPRa, etc).
- For production runs, prefer the CLI over the notebook flow:
  ```bash
  perturbflow run --config workflow/config.yaml
  ```
- The full AnnData with all PerturbFlow-added columns is in
  `notebook_results/perturbflow.h5ad` (if you used the CLI flow); load it
  with `pf.read_h5ad(...)` to keep iterating in a notebook without
  re-running the upstream stages.